In [1]:
!pip install waymo-open-dataset-tf-2-12-0==1.6.7 --no-deps -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 33.7 MB/s eta 0:00:00


In [2]:
import os
import tensorflow as tf
from waymo_open_dataset.protos import scenario_pb2
import pandas as pd

print("TF:", tf.__version__)

TF: 2.20.0


In [3]:
from google.colab import auth
auth.authenticate_user()

In [4]:
# Which shards to process. Start with 3 - takes ~6-8 minutes total.
# We can scale to more later.
SHARD_INDICES = [0, 1, 2]
TOTAL_SHARDS = 150
OUTPUT_DIR = '/content/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

OBJECT_TYPES = {0: 'unset', 1: 'vehicle', 2: 'pedestrian', 3: 'cyclist', 4: 'other'}
LANE_TYPES   = {0: 'undefined', 1: 'freeway', 2: 'surface_street', 3: 'bike_lane'}
SIGNAL_STATES = {
    0: 'unknown',     1: 'arrow_stop',  2: 'arrow_caution',
    3: 'arrow_go',    4: 'stop',        5: 'caution',
    6: 'go',          7: 'flashing_stop', 8: 'flashing_caution',
}

In [5]:
def convert_shard(shard_index):
    shard_str = f'{shard_index:05d}-of-{TOTAL_SHARDS:05d}'
    tfrecord_path = f'/content/validation_interactive.tfrecord-{shard_str}'
    gcs_path = (
        'gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/'
        f'validation_interactive/validation_interactive.tfrecord-{shard_str}'
    )

    if not os.path.exists(tfrecord_path):
        print(f"Downloading shard {shard_index}...")
        os.system(f'gcloud storage cp {gcs_path} /content/ 2>/dev/null')

    print(f"Processing shard {shard_index}...")

    scenes, agents, agent_states = [], [], []
    map_lanes, map_crosswalks, map_road_edges = [], [], []
    traffic_signals = []

    dataset = tf.data.TFRecordDataset([tfrecord_path], compression_type='')
    for i, raw in enumerate(dataset):
        s = scenario_pb2.Scenario()
        s.ParseFromString(raw.numpy())

        scene_id = s.scenario_id
        sdc_idx  = s.sdc_track_index
        ooi      = set(s.objects_of_interest)
        ttp      = set(rp.track_index for rp in s.tracks_to_predict)

        scenes.append({
            'shard_id': shard_index, 'scene_id': scene_id,
            'n_timesteps': len(s.timestamps_seconds),
            'current_time_index': s.current_time_index,
            'sdc_track_index': sdc_idx,
            'n_tracks': len(s.tracks),
            'n_map_features': len(s.map_features),
        })

        for track_idx, track in enumerate(s.tracks):
            agents.append({
                'shard_id': shard_index, 'scene_id': scene_id,
                'track_id': track.id, 'track_index': track_idx,
                'object_type': OBJECT_TYPES.get(track.object_type, 'unknown'),
                'is_sdc': track_idx == sdc_idx,
                'is_object_of_interest': track_idx in ooi,
                'is_track_to_predict':   track_idx in ttp,
            })
            for t, st in enumerate(track.states):
                agent_states.append({
                    'shard_id': shard_index, 'scene_id': scene_id,
                    'track_id': track.id, 'timestep': t,
                    'time_seconds': s.timestamps_seconds[t],
                    'center_x': st.center_x, 'center_y': st.center_y, 'center_z': st.center_z,
                    'length': st.length, 'width': st.width, 'height': st.height,
                    'heading': st.heading,
                    'velocity_x': st.velocity_x, 'velocity_y': st.velocity_y,
                    'valid': st.valid,
                })

        for feature in s.map_features:
            if feature.HasField('lane'):
                ln = feature.lane
                for k, p in enumerate(ln.polyline):
                    map_lanes.append({
                        'shard_id': shard_index, 'scene_id': scene_id,
                        'feature_id': feature.id,
                        'lane_type': LANE_TYPES.get(ln.type, 'unknown'),
                        'speed_limit_mph': ln.speed_limit_mph,
                        'point_idx': k, 'x': p.x, 'y': p.y, 'z': p.z,
                    })
            elif feature.HasField('crosswalk'):
                for k, p in enumerate(feature.crosswalk.polygon):
                    map_crosswalks.append({
                        'shard_id': shard_index, 'scene_id': scene_id,
                        'feature_id': feature.id,
                        'point_idx': k, 'x': p.x, 'y': p.y, 'z': p.z,
                    })
            elif feature.HasField('road_edge'):
                for k, p in enumerate(feature.road_edge.polyline):
                    map_road_edges.append({
                        'shard_id': shard_index, 'scene_id': scene_id,
                        'feature_id': feature.id,
                        'edge_type': int(feature.road_edge.type),
                        'point_idx': k, 'x': p.x, 'y': p.y, 'z': p.z,
                    })

        # NEW: traffic signal states per (scene, timestep, lane)
        for t, dms in enumerate(s.dynamic_map_states):
            for ls in dms.lane_states:
                traffic_signals.append({
                    'shard_id': shard_index, 'scene_id': scene_id,
                    'timestep': t,
                    'time_seconds': s.timestamps_seconds[t],
                    'lane_id': ls.lane,
                    'state': SIGNAL_STATES.get(ls.state, 'unknown'),
                    'stop_x': ls.stop_point.x,
                    'stop_y': ls.stop_point.y,
                    'stop_z': ls.stop_point.z,
                })

    suffix = f'_shard{shard_index:03d}'
    pd.DataFrame(scenes).to_parquet(f'{OUTPUT_DIR}/scenes{suffix}.parquet', index=False)
    pd.DataFrame(agents).to_parquet(f'{OUTPUT_DIR}/agents{suffix}.parquet', index=False)
    pd.DataFrame(agent_states).to_parquet(f'{OUTPUT_DIR}/agent_states{suffix}.parquet', index=False)
    pd.DataFrame(map_lanes).to_parquet(f'{OUTPUT_DIR}/map_lanes{suffix}.parquet', index=False)
    pd.DataFrame(map_crosswalks).to_parquet(f'{OUTPUT_DIR}/map_crosswalks{suffix}.parquet', index=False)
    pd.DataFrame(map_road_edges).to_parquet(f'{OUTPUT_DIR}/map_road_edges{suffix}.parquet', index=False)
    if traffic_signals:
        pd.DataFrame(traffic_signals).to_parquet(f'{OUTPUT_DIR}/traffic_signals{suffix}.parquet', index=False)

    os.remove(tfrecord_path)  # free disk for the next shard

    print(f"  Done shard {shard_index}: "
          f"{len(scenes)} scenes, "
          f"{len(agents):,} agents, "
          f"{len(agent_states):,} states, "
          f"{len(traffic_signals):,} signal rows")

In [6]:
for idx in SHARD_INDICES:
    convert_shard(idx)

print("\nAll done.")
!ls -lh /content/processed/

Processing shard 0...
  Done shard 0: 282 scenes, 19,021 agents, 1,730,911 states, 158,189 signal rows
Processing shard 1...
  Done shard 1: 294 scenes, 20,736 agents, 1,886,976 states, 167,978 signal rows
Processing shard 2...
  Done shard 2: 274 scenes, 18,319 agents, 1,667,029 states, 152,315 signal rows

All done.
total 404M
-rw-r--r-- 1 root root  67K May  7 11:12 agents_shard000.parquet
-rw-r--r-- 1 root root  72K May  7 11:13 agents_shard001.parquet
-rw-r--r-- 1 root root  58K May  7 11:13 agents_shard002.parquet
-rw-r--r-- 1 root root  31M May  7 11:12 agent_states_shard000.parquet
-rw-r--r-- 1 root root  32M May  7 11:13 agent_states_shard001.parquet
-rw-r--r-- 1 root root  29M May  7 11:13 agent_states_shard002.parquet
-rw-r--r-- 1 root root 223K May  7 11:12 map_crosswalks_shard000.parquet
-rw-r--r-- 1 root root 230K May  7 11:13 map_crosswalks_shard001.parquet
-rw-r--r-- 1 root root 191K May  7 11:14 map_crosswalks_shard002.parquet
-rw-r--r-- 1 root root  70M May  7 11:12 m

In [7]:
import shutil
shutil.make_archive('/content/processed_v2', 'zip', '/content/processed')
from google.colab import files
files.download('/content/processed_v2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>